# Transformando Nuestra Aplicación Full-Stack en una Aplicación RAG Multimodal

## Cómo Actualizar la App de Preguntas sobre PDFs para Entender Tanto Texto COMO Imágenes

En nuestros notebooks anteriores, construimos dos cosas:

1. **Una App Full-Stack de PDFs** (carpetas `052-pdfapp-LC-backend` y `053-pdfapp-LC-frontend`) que permite a los usuarios subir PDFs, almacenarlos en AWS S3, y hacer preguntas sobre ellos usando un Agente RAG con LangChain 1.0.

2. **Una App RAG Multimodal** (notebook `ZZZ-MULTIMODAL-APP-WITH-LC1`) que puede procesar PDFs que contienen texto, tablas E imágenes.

**Ahora la gran pregunta es: ¿Cómo combinamos estas dos?** ¿Cómo tomamos nuestra app full-stack y la actualizamos para que pueda entender imágenes dentro de los PDFs, no solo texto?

Este notebook explicará paso a paso, en términos sencillos, exactamente qué cambios se necesitan.

---
## Tabla de Contenidos

1. La Visión General: ¿Qué Necesita Cambiar?
2. Entendiendo la App Actual (RAG Regular)
3. Entendiendo el Objetivo (RAG Multimodal)
4. ¡La Buena Noticia: El Frontend Apenas Cambia!
5. Cambio en el Backend 1: Nuevas Dependencias
6. Cambio en el Backend 2: Reemplazar el Cargador de PDFs
7. Cambio en el Backend 3: Añadir Resumen de Imágenes con GPT-4o
8. Cambio en el Backend 4: Reemplazar el Vector Store
9. Cambio en el Backend 5: Actualizar el Agente RAG
10. Cambio en el Backend 6: Reescribir el Endpoint `/ask`
11. Comparación Completa del Código Antes y Después
12. Resumen de Todos los Cambios

---
<a id='section1'></a>
## 1. La Visión General: ¿Qué Necesita Cambiar?

Aquí está lo más importante que debes entender:

> **El frontend se mantiene casi igual. Todos los cambios reales ocurren en el backend.**

¿Por qué? Porque el trabajo del frontend es simple: enviar una pregunta, recibir una respuesta, mostrarla. No le importa si el backend usó RAG regular o RAG multimodal para encontrar esa respuesta. La "magia" de entender imágenes ocurre completamente en el lado del servidor.

### Analogía del Restaurante

¿Recuerdas nuestra analogía del restaurante del notebook anterior?

- **Antes (RAG Regular)**: La cocina (backend) solo podía leer el texto de las recetas. Si una receta tenía una foto mostrando cómo debería verse el plato, la cocina la ignoraba.

- **Después (RAG Multimodal)**: La cocina ahora tiene un **consultor fotógrafo de comida** (visión de GPT-4o) que puede mirar las fotos y describir lo que muestran. ¡Ahora cuando un cliente pregunta "¿Cómo se ve este plato?", la cocina puede responder!

El comedor (frontend) no cambia. El camarero sigue llevando preguntas y respuestas de un lado a otro de la misma manera.

### Visión General de los Cambios

```
┌──────────────────────────────────────────────────────────────────────────┐
│                        ¿QUÉ CAMBIA?                                     │
│                                                                          │
│   Frontend (Next.js)          Backend (FastAPI)                          │
│   ┌─────────────────┐         ┌──────────────────────────────────────┐   │
│   │                 │         │                                      │   │
│   │  ¡NO SE NECESITAN│         │  MUCHOS CAMBIOS:                    │   │
│   │  CAMBIOS!       │         │                                      │   │
│   │                 │         │  1. Nuevas dependencias              │   │
│   │  Misma entrada  │         │  2. Extracción de texto/tablas (API) │   │
│   │  de preguntas,  │         │  3. Renderizado de páginas (pymupdf) │   │
│   │  misma muestra  │         │  4. Resumen de imágenes (GPT-4o)    │   │
│   │  de respuestas  │         │  5. Multi-Vector Retriever          │   │
│   │                 │         │  6. Agente RAG actualizado           │   │
│   │                 │         │  7. Endpoint /ask reescrito          │   │
│   │                 │         │  8. config.py (nueva clave API)      │   │
│   │                 │         │                                      │   │
│   └─────────────────┘         └──────────────────────────────────────┘   │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘
```

---
<a id='section2'></a>
## 2. Entendiendo la App Actual (RAG Regular)

Primero entendamos cómo la app actual procesa un PDF cuando un usuario hace una pregunta.

### Flujo Actual (RAG Solo Texto)

Cuando un usuario pulsa "Preguntar" en un PDF, esto es lo que ocurre en el backend (`routers/pdfs.py`):

```
El usuario hace una pregunta
      │
      ▼
1. Descargar PDF desde S3
      │
      ▼
2. Cargar PDF con PyPDFLoader          ◄── Solo lee TEXTO
      │
      ▼
3. Dividir texto en fragmentos         ◄── División simple de texto
      │
      ▼
4. Crear embeddings                    ◄── Embeddings de fragmentos de texto sin procesar
      │
      ▼
5. Almacenar en InMemoryVectorStore    ◄── Vector store simple
      │
      ▼
6. El agente busca y responde          ◄── Solo tiene texto con el que trabajar
```

### El Código Actual

Aquí está la función actual `create_pdf_rag_agent` de `routers/pdfs.py`:

```python
def create_pdf_rag_agent(pdf_path: str):
    # Cargar el PDF (SOLO TEXTO)
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # Dividir en fragmentos
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        add_start_index=True
    )
    all_splits = text_splitter.split_documents(documents)

    # Crear embeddings y vector store
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(documents=all_splits)

    # Crear una herramienta de búsqueda
    @tool
    def search_pdf(query: str) -> str:
        """Search the PDF document for relevant information."""
        results = vector_store.similarity_search(query, k=3)
        return "\n\n".join([doc.page_content for doc in results])

    # Crear el agente
    agent = create_agent(
        model="gpt-4o-mini",
        tools=[search_pdf],
        system_prompt="""You are a helpful assistant that answers questions
        about a PDF document. Use the search_pdf tool to find relevant
        information in the document."""
    )

    return agent
```

### ¿Cuál es el Problema?

El problema está en el **Paso 2**: `PyPDFLoader` solo extrae texto. Si tu PDF tiene:

| Tipo de Contenido | ¿Lo Extrae PyPDFLoader? |
|-------------------|------------------------|
| Texto simple      | Sí |
| Tablas            | Parcialmente (pierde la estructura) |
| Imágenes          | **¡No! Completamente ignoradas** |
| Gráficos          | **¡No! Completamente ignorados** |
| Diagramas         | **¡No! Completamente ignorados** |

Así que si un usuario pregunta "¿Qué muestra el gráfico de la página 3?", la app actual **no puede responder** porque ¡nunca vio el gráfico!

---
<a id='section3'></a>
## 3. Entendiendo el Objetivo (RAG Multimodal)

Ahora veamos qué hace de manera diferente la versión multimodal.

### Flujo Objetivo (RAG Multimodal)

```
El usuario hace una pregunta
      |
      v
1. Descargar PDF desde S3
      |
      v
2a. Enviar PDF a la API de Unstructured     <-- Lee TEXTO + TABLAS
      |
      +-- Elementos de texto
      +-- Elementos de tabla
      |
2b. Renderizar páginas con pymupdf          <-- Captura TODO visualmente
      |
      +-- Imágenes de páginas (como base64)
      |
      v
3. Resumir todo (GPT-4o-mini)               <-- ¡NUEVO PASO!
      |
      +-- Texto    -> Resumido
      +-- Tablas   -> Resumidas
      +-- Imágenes de páginas -> Descritas
      |
      v
4. Multi-Vector Retriever                   <-- Vector store MEJORADO
      |
      +-- Vector Store: almacena RESÚMENES (para buscar)
      +-- Doc Store: almacena ORIGINALES (para responder)
      |
      v
5. El agente busca y responde               <-- ¡Ahora tiene texto + tablas + imágenes!
```

### ¿Por Qué Dos Herramientas para la Extracción de PDFs?

Usamos un **enfoque de dos herramientas** porque cada herramienta es mejor en cosas diferentes:

| Herramienta | Mejor En | Se Usa Para |
|-------------|----------|-------------|
| **API de Unstructured** | Extraer texto estructurado y tablas | Extracción de texto y tablas |
| **pymupdf** | Renderizar páginas completas como imágenes | Capturar gráficos, diagramas, fotos — todo lo visual |

La API de Unstructured es excelente para analizar texto y tablas, pero puede pasar por alto imágenes — especialmente gráficos basados en vectores (dibujados con formas, no incrustados como archivos de imagen). Al renderizar cada página como una imagen con pymupdf, garantizamos que GPT-4o-mini vea **todo** en la página.

### Las Diferencias Clave

| Aspecto | RAG Regular (Actual) | RAG Multimodal (Objetivo) |
|---------|---------------------|---------------------------|
| **Procesamiento de Texto/Tablas** | `PyPDFLoader` (local) | `UnstructuredClient` (API alojada) |
| **Procesamiento de Imágenes** | Ninguno | Renderizado de páginas con `pymupdf` + visión de GPT-4o-mini |
| **Qué se extrae** | Solo texto | Texto + Tablas + Imágenes de páginas completas |
| **Resumen** | Ninguno | GPT-4o-mini para todos los tipos de contenido |
| **Vector Store** | `InMemoryVectorStore` | `MultiVectorRetriever` con Chroma |
| **Qué se puede buscar** | Fragmentos de texto sin procesar | Resúmenes de texto, tablas Y descripciones de páginas |
| **Preguntas sobre imágenes** | No puede responder | ¡Puede responder! |
| **Dependencias** | `pypdf`, `langchain` | `unstructured-client`, `chromadb`, `pillow`, `pymupdf` |
| **Dependencias del sistema** | Ninguna | Ninguna (todas instalables con pip) |

---
<a id='section4'></a>
## 4. ¡La Buena Noticia: El Frontend Apenas Cambia!

Esta es la mejor parte. Veamos por qué el frontend no necesita cambios.

### El Trabajo del Frontend

El frontend (`components/pdf.js`) solo hace 3 cosas:

1. **Recoge** la pregunta del usuario (un campo de texto)
2. **La envía** al backend (`POST /pdfs/{id}/ask`)
3. **Muestra** la respuesta (un párrafo de texto)

```javascript
// Esto es lo que envía el frontend:
{
    "question": "¿Qué muestra el gráfico de la página 3?"
}

// Esto es lo que recibe el frontend:
{
    "pdf_id": 5,
    "pdf_name": "informe-financiero.pdf",
    "question": "¿Qué muestra el gráfico de la página 3?",
    "answer": "El gráfico de la página 3 muestra un crecimiento de ingresos del 25% interanual..."
}
```

Fíjate: ¡el formato de la solicitud y la respuesta es **exactamente el mismo** ya sea que usemos RAG regular o RAG multimodal! Al frontend no le importa cómo encontró la respuesta el backend.

### Analogía: Pedir en un Restaurante

Imagina que estás en un restaurante y le preguntas al camarero:

> "¿Qué ingredientes lleva el especial del chef?"

El camarero va a la cocina y vuelve con la respuesta. **No te importa si el chef:**
- Leyó el texto de la receta
- Miró una foto del plato
- Analizó un gráfico de información nutricional

Solo te importa obtener una buena respuesta. ¡Lo mismo con nuestro frontend!

### El Único Cambio Opcional del Frontend

Si quisieras ser amable con el usuario, podrías actualizar el texto del placeholder para sugerir que pueden preguntar sobre imágenes:

```javascript
// ANTES (actual)
placeholder="Haz una pregunta sobre este PDF..."

// DESPUÉS (mejora opcional)
placeholder="Pregunta sobre texto, tablas o imágenes en este PDF..."
```

Pero esto es puramente estético. La app funciona perfectamente sin ello.

---
<a id='section5'></a>
## 5. Cambio en el Backend 1: Nuevas Dependencias

El primer cambio es añadir nuevas bibliotecas de Python que habilitan el procesamiento multimodal.

### Qué Añadir a `pyproject.toml`

```toml
# ANTES (dependencias actuales)
dependencies = [
    "fastapi (>=0.128.0,<0.129.0)",
    "uvicorn[standard] (>=0.40.0,<0.41.0)",
    "alembic (>=1.18.2,<2.0.0)",
    "psycopg2-binary (>=2.9.11,<3.0.0)",
    "pydantic-settings (>=2.12.0,<3.0.0)",
    "boto3 (>=1.42.37,<2.0.0)",
    "python-multipart (>=0.0.22,<0.0.23)",
    "langchain (>=1.0.0,<2.0.0)",
    "langchain-openai (>=1.0.0,<2.0.0)",
    "langchain-community (>=0.4.0,<1.0.0)",
    "langchain-text-splitters (>=1.0.0,<2.0.0)",
    "pypdf (>=5.0.0,<6.0.0)",
    "requests (>=2.32.0,<3.0.0)"
]

# DESPUÉS (con las nuevas dependencias multimodales)
dependencies = [
    # ... mantener todas las dependencias existentes ...
    # AÑADIR estas nuevas:
    "langchain-chroma (>=1.1.0,<2.0.0)",          # Base de datos vectorial para Multi-Vector Retriever
    "pillow (>=12.1.0,<13.0.0)",                   # Procesamiento de imágenes
    "unstructured-client (>=0.42.10,<0.43.0)",     # Cliente API para el servicio alojado de Unstructured
    "langchain-unstructured (>=1.0.1,<2.0.0)",     # Integración de LangChain con Unstructured
    "pymupdf (>=1.25.0,<2.0.0)",                   # Renderiza páginas PDF como imágenes para GPT-4o
]
```

### ¡No Se Necesitan Dependencias del Sistema!

Todas las nuevas dependencias son paquetes instalables puramente con pip:

- La **API de Unstructured** envía tu PDF a los servidores de Unstructured para la extracción de texto/tablas — no se necesita `poppler` ni `tesseract` local.
- **pymupdf** incluye su propio motor de renderizado de PDF — no se requieren bibliotecas PDF a nivel del sistema.

Esto significa:
- No necesitas `brew install poppler tesseract` en macOS
- No necesitas `sudo apt install poppler-utils tesseract-ocr` en Linux
- Configuración más sencilla, funciona igual en todas las máquinas

### Variable de Entorno

Necesitarás una **clave API de Unstructured** en tu archivo `.env`:

```
UNSTRUCTURED_API_KEY=tu_clave_api_aquí
```

Puedes obtener una clave API gratuita en [unstructured.io](https://unstructured.io).

### ¿Por Qué Cada Nueva Dependencia?

| Paquete | Por Qué Lo Necesitamos |
|---------|------------------------|
| `unstructured-client` | Cliente API que envía PDFs al servicio alojado de Unstructured para la extracción de texto y tablas |
| `langchain-unstructured` | Capa de integración de LangChain para la API de Unstructured |
| `langchain-chroma` | Proporciona la base de datos vectorial Chroma necesaria para el `MultiVectorRetriever` |
| `pillow` | Maneja el procesamiento de imágenes |
| `pymupdf` | Renderiza páginas PDF como imágenes (PNG) para que GPT-4o pueda ver todo en cada página — gráficos, diagramas, fotos, etc. |

---
<a id='section6'></a>
## 6. Cambio en el Backend 2: Reemplazar el Cargador de PDFs

Este es el cambio más fundamental. Reemplazamos el simple `PyPDFLoader` con un **enfoque de dos herramientas**: la API de Unstructured para texto/tablas, y pymupdf para imágenes de páginas.

### ANTES: PyPDFLoader (Solo Texto)

```python
from langchain_community.document_loaders import PyPDFLoader

# Solo extrae texto - ¡ignora las imágenes!
loader = PyPDFLoader(pdf_path)
documents = loader.load()
```

### DESPUÉS: API de Unstructured + pymupdf (Texto + Tablas + Imágenes de Páginas)

```python
import base64
import fitz  # pymupdf
from unstructured_client import UnstructuredClient
from config import get_settings

def extract_pdf_elements(pdf_path: str):
    """
    Extraer texto, tablas e imágenes de un archivo PDF.

    - Texto y tablas: extraídos mediante la API de Unstructured
    - Imágenes de páginas: renderizadas mediante pymupdf (garantiza que GPT-4o vea todo)

    ¿Por qué dos herramientas? La API de Unstructured es excelente para texto estructurado y tablas,
    pero puede pasar por alto imágenes — especialmente gráficos basados en vectores dibujados con formas.
    pymupdf renderiza cada página como una imagen, así que nada se pierde.
    """
    settings = get_settings()
    client = UnstructuredClient(api_key_auth=settings.UNSTRUCTURED_API_KEY)

    with open(pdf_path, "rb") as f:
        file_content = f.read()

    # Usar la API de Unstructured para extracción de texto y tablas
    response = client.general.partition(
        request={
            "partition_parameters": {
                "files": {
                    "content": file_content,
                    "file_name": os.path.basename(pdf_path),
                },
                "strategy": "hi_res",
                "infer_table_structure": True,
            }
        }
    )

    # Sin fragmentación, la API devuelve tipos de elementos granulares
    text_types = {
        "NarrativeText", "Title", "UncategorizedText",
        "ListItem", "Header", "Footer", "FigureCaption",
    }

    raw_texts = []
    table_elements = []

    for element in response.elements:
        el_type = element.get("type", "")
        text = element.get("text", "")

        if el_type in text_types and text.strip():
            raw_texts.append(text)
        elif el_type == "Table":
            table_elements.append(text)

    # Agrupar elementos de texto pequeños en fragmentos de ~2000 caracteres
    text_elements = []
    current_chunk = ""
    for text in raw_texts:
        if len(current_chunk) + len(text) > 2000 and current_chunk:
            text_elements.append(current_chunk.strip())
            current_chunk = text
        else:
            current_chunk += "\n\n" + text if current_chunk else text
    if current_chunk.strip():
        text_elements.append(current_chunk.strip())

    # Renderizar cada página del PDF como imagen usando pymupdf
    image_base64_list = []
    doc = fitz.open(pdf_path)
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        img_b64 = base64.b64encode(img_bytes).decode("utf-8")
        image_base64_list.append(img_b64)
    doc.close()

    return text_elements, table_elements, image_base64_list
```

### ¿Qué Cambió y Por Qué?

| Antes | Después | Por Qué |
|-------|---------|----------|
| `PyPDFLoader` (local, solo texto) | `UnstructuredClient` (API) | Puede extraer tablas con la estructura preservada |
| Sin extracción de imágenes | `pymupdf` renderiza páginas como imágenes | GPT-4o ve todo: gráficos, fotos, diagramas |
| Devuelve una lista de documentos | Devuelve tres listas separadas | Cada tipo de contenido necesita un procesamiento diferente |
| Fragmentación básica de texto | Agrupación local en fragmentos de ~2000 caracteres | No se necesita fragmentación del lado de la API |

### ¿Por Qué NO Usar la API de Unstructured para Extracción de Imágenes?

Te podrías preguntar: la API de Unstructured tiene un parámetro `extract_image_block_types` — ¿por qué no usarlo?

En la práctica, este parámetro tiene **dos problemas**:

1. **Los gráficos basados en vectores no se detectan**: Muchos gráficos de PDF se dibujan con formas vectoriales (rectángulos, líneas), no se incrustan como imágenes raster. La API no los detecta como elementos "Image".

2. **Conflictos con la fragmentación**: Si habilitas `chunking_strategy` (para agrupar texto), los elementos Image se absorben en los fragmentos `CompositeElement` y sus datos en base64 se pierden. Pero si deshabilitas la fragmentación, obtienes muchos fragmentos de texto muy pequeños.

El **enfoque de pymupdf** resuelve ambos problemas: renderiza cada página como una imagen de píxeles, así que GPT-4o ve todo — gráficos vectoriales, fotos raster, diagramas, superposiciones de texto, todo.

### Cómo Funciona

```
Tu Backend
     │
     ├──► API de Unstructured
     │         │
     │         ├── Extrae elementos de texto (NarrativeText, Title, etc.)
     │         └── Extrae tablas (con estructura)
     │
     └──► pymupdf (local)
               │
               └── Renderiza cada página a 200 DPI → PNG → base64
                   (captura gráficos, fotos, diagramas, todo)
```

### Analogía: El Inspector de Cocina

- **Antes (PyPDFLoader)**: Un inspector de cocina que solo puede leer el texto de la receta. Si hay una foto del plato, la ignora.

- **Después (API de Unstructured + pymupdf)**: Envías la receta a un laboratorio de análisis de texto (API de Unstructured) para el contenido escrito, Y tomas una foto de alta resolución de cada página (pymupdf) para que un experto visual (GPT-4o) pueda examinar todo lo que aparece en la página.

---
<a id='section7'></a>
## 7. Cambio en el Backend 3: Añadir Resumen de Imágenes con GPT-4o-mini

¡Este es el cambio **más emocionante**! Ahora tenemos imágenes de páginas renderizadas por pymupdf (como cadenas PNG en base64), y necesitamos que la IA las "mire" y las describa.

### ¿Por Qué No Podemos Simplemente Almacenar las Imágenes Directamente?

¡Gran pregunta! Aquí está el problema:

- Nuestro vector store trabaja con **embeddings de texto** (números que representan el significado del texto)
- Las imágenes son **píxeles**, no texto
- No podemos crear embeddings de texto a partir de píxeles

**La solución**: Pedirle a GPT-4o-mini (¡que puede ver imágenes!) que **describa** cada imagen de página en texto. Luego almacenamos esa descripción en texto.

### El Nuevo Código de Resumen de Imágenes

pymupdf renderiza cada página como una imagen PNG y la codifica en base64. Se la pasamos directamente a GPT-4o-mini:

```python
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage


def summarize_image(image_base64: str, model: ChatOpenAI) -> str:
    """
    Usar un modelo con capacidad de visión para describir una imagen de página.

    El parámetro image_base64 proviene del renderizado de páginas de pymupdf
    (página -> pixmap -> PNG -> base64).
    """
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": """Describe this image in detail. Include:
                - What type of content it shows (chart, diagram, photo, etc.)
                - Any text visible in the image
                - Key data points if it's a chart or graph
                - The overall meaning or purpose of the image"""
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{image_base64}"
                }
            }
        ]
    )

    response = model.invoke([message])
    return response.content
```

### Cómo Funciona Paso a Paso

```
pymupdf renderiza cada página del PDF como una imagen PNG
        |
        v
1. Cada página se renderiza a 200 DPI -> bytes PNG -> cadena base64
        |
        v
2. Se envía a GPT-4o-mini con el prompt: "Describe esta imagen..."
        |
        v
3. GPT-4o-mini "ve" la página completa y devuelve una descripción en texto:
   "Esta página muestra un estado financiero con un gráfico de barras
    que muestra las ventas de 2020 a 2024. El gráfico muestra:
    2020: ~1M$, 2021: ~3M$, 2022: ~8M$, 2023: ~15M$, 2024: ~22M$.
    También hay una foto de una GPU y una insignia de ROI del 33%..."
        |
        v
4. Esta descripción en TEXTO se almacena en el vector store
```

Este enfoque es más fiable que depender de la API de Unstructured para extraer imágenes individuales, porque:
- pymupdf captura **todo** en la página (gráficos vectoriales, fotos, diagramas, texto)
- Ningún elemento se pierde debido a la fragmentación o problemas de clasificación de tipos de elementos
- GPT-4o-mini ve la página exactamente como la vería un humano

### También Resumimos el Texto y las Tablas

En la versión multimodal, resumimos TODOS los tipos de contenido. Esto se debe a que usamos el patrón **Multi-Vector Retriever** donde los resúmenes se usan para buscar, y los originales se usan para responder:

```python
def summarize_text(text: str, model: ChatOpenAI) -> str:
    """Resumir un elemento de texto."""
    prompt = f"""Summarize the following text concisely 
    while preserving key information:\n\n{text}\n\nSummary:"""
    response = model.invoke([HumanMessage(content=prompt)])
    return response.content


def summarize_table(table: str, model: ChatOpenAI) -> str:
    """Resumir un elemento de tabla."""
    prompt = f"""Summarize the following table, highlighting 
    key data points and relationships:\n\n{table}\n\nSummary:"""
    response = model.invoke([HumanMessage(content=prompt)])
    return response.content
```

### Un Modelo para Todo: GPT-4o-mini

Usamos **`gpt-4o-mini` para todos los tipos de contenido** — texto, tablas e imágenes:

| Tipo de Contenido | Modelo | Por Qué |
|-------------------|--------|----------|
| Texto | `gpt-4o-mini` | Rápido, económico y más capaz que el antiguo `gpt-3.5-turbo` |
| Tablas | `gpt-4o-mini` | Mejor fiabilidad para datos estructurados que `gpt-3.5-turbo` |
| Imágenes de páginas | `gpt-4o-mini` | Soporte completo de visión con excelente relación coste/calidad |

### ¿Por Qué No GPT-3.5-turbo?

Puede que veas tutoriales antiguos que usan `gpt-3.5-turbo` para el resumen de texto. **OpenAI ahora lo trata como obsoleto** y recomienda `gpt-4o-mini` en su lugar — es más capaz, multimodal (puede manejar texto E imágenes) y ofrece una velocidad similar con mejor relación calidad-precio.

Usar un solo modelo para todo también **simplifica el código** — no hay necesidad de variables separadas `text_model` y `vision_model`.

### ¿Quieres Mayor Calidad para las Imágenes?

Si necesitas descripciones de imágenes más detalladas para documentos complejos, puedes mejorar solo el paso de imágenes a `gpt-4o`:

```python
# Por defecto: un modelo para todo
model = ChatOpenAI(model="gpt-4o-mini", max_tokens=1024)

# Opcional: usar gpt-4o para imágenes si necesitas mayor calidad
# vision_model = ChatOpenAI(model="gpt-4o", max_tokens=1024)
```

### Nota Importante Sobre los Modelos de Visión

Puede que veas tutoriales antiguos que mencionan `gpt-4-vision-preview`. **¡Ese modelo está obsoleto!** Las capacidades de visión ahora están integradas en muchos modelos, incluyendo `gpt-4o-mini` y `gpt-4o`.

---
<a id='section8'></a>
## 8. Cambio en el Backend 4: Reemplazar el Vector Store

La app actual usa un simple `InMemoryVectorStore`. La versión multimodal necesita un `MultiVectorRetriever` más sofisticado.

### ¿Por Qué Necesitamos un Vector Store Diferente?

En RAG regular:
- Almacenamos **fragmentos de texto sin procesar** y los buscamos directamente
- El mismo texto que buscamos es el mismo texto que usamos para responder

En RAG multimodal, tenemos un **patrón de dos almacenes**:
- **Almacén 1 (Vector Store)**: Contiene **resúmenes** (para buscar)
- **Almacén 2 (Doc Store)**: Contiene **contenido original** (para responder)

```
┌─────────────────────────────┐    ┌─────────────────────────────┐
│    VECTOR STORE (Chroma)    │    │    DOC STORE (InMemory)     │
│                             │    │                             │
│  Almacena: RESÚMENES        │    │  Almacena: ORIGINALES       │
│  (como embeddings)          │    │  (contenido completo)       │
│                             │    │                             │
│  "Los ingresos crecieron    │◄──►│  Tabla completa con todos   │
│   un 25%..."                │ ID │  los datos                  │
│  UUID: abc-123              │    │  UUID: abc-123              │
│                             │    │                             │
│  "Gráfico de barras que     │◄──►│  Descripción completa de    │
│   muestra..."               │ ID │  la imagen                  │
│  UUID: def-456              │    │  UUID: def-456              │
└─────────────────────────────┘    └─────────────────────────────┘
```

### ¿Por Qué Dos Almacenes?

- **Los resúmenes son mejores para buscar**: Un resumen conciso captura el significado clave
- **Los originales son mejores para responder**: El contenido completo tiene todos los detalles

Es como el catálogo de fichas de una biblioteca:
- La **ficha del catálogo** (resumen) te ayuda a encontrar el libro correcto
- Pero lees el **libro real** (original) para obtener la respuesta detallada

### ANTES: Vector Store Simple

```python
from langchain_core.vectorstores import InMemoryVectorStore

# Simple: almacenar fragmentos sin procesar, buscar fragmentos sin procesar
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents=all_splits)
```

### DESPUÉS: Multi-Vector Retriever

```python
import uuid
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.schema.document import Document
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma


def create_multimodal_retriever(
    text_summaries, text_elements,
    table_summaries, table_elements,
    image_summaries
):
    # Crear el modelo de embeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # Crear los dos almacenes
    vectorstore = Chroma(
        collection_name="multimodal_summaries",
        embedding_function=embeddings
    )
    docstore = InMemoryStore()
    id_key = "doc_id"
    
    # Crear el multi-vector retriever
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        id_key=id_key
    )
    
    # Función auxiliar para añadir documentos
    def add_documents(summaries, original_contents, content_type):
        if not summaries:
            return
        doc_ids = [str(uuid.uuid4()) for _ in summaries]
        summary_docs = [
            Document(
                page_content=s,
                metadata={id_key: doc_ids[i], "type": content_type}
            )
            for i, s in enumerate(summaries)
        ]
        retriever.vectorstore.add_documents(summary_docs)
        retriever.docstore.mset(list(zip(doc_ids, original_contents)))
    
    # Añadir todos los tipos de contenido
    add_documents(text_summaries, text_elements, "text")
    add_documents(table_summaries, table_elements, "table")
    # Para imágenes, el resumen ES el contenido (la descripción)
    add_documents(image_summaries, image_summaries, "image")
    
    return retriever
```

### Entendiendo los UUIDs

Cada pieza de contenido recibe un **UUID** (un identificador único) que vincula su resumen con su original:

```
Resumen: "Gráfico que muestra crecimiento de ingresos del 25%"   ──── UUID: abc-123
                                                                           │
Original: "Este gráfico de barras muestra los ingresos            ──── UUID: abc-123
           trimestrales del año fiscal 2025. T1: 2,3M$,
           T2: 2,8M$, T3: 3,1M$, T4: 3,5M$..."
```

Cuando una búsqueda encuentra el resumen, el UUID se usa para obtener el contenido original completo.

### Nota Importante: `langchain_classic`

En LangChain 1.x, varios módulos (`retrievers`, `storage`, `schema`) se movieron de `langchain` a `langchain_classic`. Por eso importamos desde `langchain_classic` en lugar de `langchain` para estos tres:

| Import | Paquete |
|--------|----------|
| `MultiVectorRetriever` | `langchain_classic.retrievers.multi_vector` |
| `Document` | `langchain_classic.schema.document` |
| `InMemoryStore` | `langchain_classic.storage` |

---
<a id='section9'></a>
## 9. Cambio en el Backend 5: Actualizar el Agente RAG

El agente necesita una pequeña actualización en su herramienta de búsqueda y en el prompt del sistema para que sepa sobre el contenido multimodal.

### ANTES: Agente de Solo Texto

```python
# Agente actual - solo sabe sobre texto
@tool
def search_pdf(query: str) -> str:
    """Search the PDF document for relevant information."""
    results = vector_store.similarity_search(query, k=3)
    return "\n\n".join([doc.page_content for doc in results])

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_pdf],
    system_prompt="""You are a helpful assistant that answers questions
    about a PDF document. Use the search_pdf tool to find relevant
    information in the document."""
)
```

### DESPUÉS: Agente Multimodal

```python
# Agente actualizado - sabe sobre texto, tablas E imágenes
@tool
def search_documents(query: str) -> str:
    """
    Search the document database for relevant information.
    The database contains text, tables, and image descriptions
    from PDF documents.
    """
    results = retriever.invoke(query)     # ¡Usar retriever.invoke() en su lugar!
    
    if not results:
        return "No relevant information found."
    
    formatted_results = []
    for i, result in enumerate(results, 1):
        if hasattr(result, 'page_content'):
            formatted_results.append(f"[Result {i}]\n{result.page_content}")
        else:
            formatted_results.append(f"[Result {i}]\n{result}")
    
    return "\n\n".join(formatted_results)

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_documents],
    system_prompt="""You are a helpful assistant that answers questions
    about documents.

    You have access to a document database that contains:
    - Text content from PDF documents
    - Table data (financial figures, statistics, etc.)
    - Descriptions of images (charts, diagrams, photos)

    When answering questions:
    1. Use the search_documents tool to find relevant information
    2. Base your answers ONLY on the retrieved information
    3. If the information isn't in the documents, say so clearly
    4. For numerical questions, quote the exact figures
    5. For image-related questions, refer to the image descriptions

    Be concise but accurate."""
)
```

### ¿Qué Cambió?

| Aspecto | Antes | Después |
|---------|-------|----------|
| Nombre de la herramienta | `search_pdf` | `search_documents` |
| Método de búsqueda | `vector_store.similarity_search()` | `retriever.invoke()` |
| Descripción de la herramienta | "Search the PDF document" | Menciona texto, tablas Y descripciones de imágenes |
| Prompt del sistema | Básico | Instrucciones detalladas para contenido multimodal |
| Manejo de resultados | Simple join | Verifica diferentes tipos de resultados |

### ¿Por Qué `retriever.invoke()` en Lugar de `similarity_search()`?

- `similarity_search()` devuelve los resúmenes que coinciden
- `retriever.invoke()` busca en los resúmenes pero **devuelve los originales** (mediante UUIDs)
- Esto le da al agente contenido más rico y detallado sobre el cual basar sus respuestas

---
<a id='section10'></a>
## 10. Cambio en el Backend 6: Reescribir el Endpoint `/ask`

Finalmente, necesitamos actualizar el endpoint `/ask` en `routers/pdfs.py` para usar todas las nuevas funciones multimodales.

### ANTES: Endpoint `/ask` Actual

```python
@router.post("/{id}/ask")
async def ask_pdf(id: int, request: schemas.QuestionRequest, db: Session = Depends(get_db)):
    # Obtener el PDF de la base de datos
    pdf = crud.read_pdf(db, id)
    if pdf is None:
        raise HTTPException(status_code=404, detail="PDF not found")

    temp_path = None
    try:
        # Descargar PDF desde S3
        temp_path = download_pdf_from_url(pdf.file)

        # Crear agente RAG (solo texto)
        agent = create_pdf_rag_agent(temp_path)

        # Hacer la pregunta
        response = agent.invoke(
            {"messages": [HumanMessage(content=request.question)]}
        )
        answer = response["messages"][-1].content

        return {
            "pdf_id": id,
            "pdf_name": pdf.name,
            "question": request.question,
            "answer": answer
        }
    finally:
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)
```

### DESPUÉS: Endpoint `/ask` Multimodal

```python
@router.post("/{id}/ask")
async def ask_pdf(id: int, request: schemas.QuestionRequest, db: Session = Depends(get_db)):
    # Obtener el PDF de la base de datos
    pdf = crud.read_pdf(db, id)
    if pdf is None:
        raise HTTPException(status_code=404, detail="PDF not found")

    temp_path = None
    try:
        # Paso 1: Descargar PDF desde S3
        temp_path = download_pdf_from_url(pdf.file)

        # Paso 2: Extraer texto, tablas E imágenes mediante la API de Unstructured
        text_elements, table_elements, image_base64_list = extract_pdf_elements(
            temp_path
        )

        # Paso 3: Crear resúmenes para todos los tipos de contenido
        text_summaries, table_summaries, image_summaries = create_all_summaries(
            text_elements, table_elements, image_base64_list
        )

        # Paso 4: Construir el Multi-Vector Retriever
        retriever = create_multimodal_retriever(
            text_summaries, text_elements,
            table_summaries, table_elements,
            image_summaries
        )

        # Paso 5: Crear el agente multimodal
        agent = create_multimodal_rag_agent(retriever)

        # Paso 6: Hacer la pregunta
        response = agent.invoke(
            {"messages": [HumanMessage(content=request.question)]}
        )
        answer = response["messages"][-1].content

        return {
            "pdf_id": id,
            "pdf_name": pdf.name,
            "question": request.question,
            "answer": answer
        }
    finally:
        # Limpiar archivo PDF temporal
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)
```

### Diferencias Clave

| Paso | Antes | Después |
|------|-------|----------|
| Procesamiento del PDF | `PyPDFLoader` (solo texto) | `extract_pdf_elements` mediante la API de Unstructured (texto + tablas + imágenes) |
| Resumen | Ninguno | `create_all_summaries` (GPT-3.5 + GPT-4o) |
| Vector Store | `InMemoryVectorStore` | `create_multimodal_retriever` |
| Agente | Agente de solo texto | Agente con capacidad multimodal |
| Limpieza | Eliminar PDF temporal | ¡Solo eliminar PDF temporal (no se necesita directorio temporal de imágenes)! |

Fíjate en lo **más sencilla que es la limpieza** con el enfoque de la API. Dado que las imágenes se devuelven como cadenas base64 en la respuesta de la API (no se guardan en disco), no necesitamos crear ni limpiar un directorio temporal de imágenes.

### ¿Qué Pasa con el Rendimiento?

La versión multimodal es **más lenta** porque hace más trabajo:

| Paso | RAG Regular | RAG Multimodal | Tiempo Extra |
|------|-------------|----------------|---------------|
| Extracción del PDF | ~1 seg (local) | ~5-15 seg (llamada API) | Ida y vuelta de red + procesamiento del servidor |
| Resumen | Ninguno | ~10-30 seg | Llamar a GPT para cada elemento |
| Vector Store | ~1 seg | ~2-3 seg | Más documentos para crear embeddings |
| Consulta del agente | ~3-5 seg | ~3-5 seg | Igual |
| **Total** | **~5-7 seg** | **~20-55 seg** | ¡Vale la pena para multimodal! |

**Consejo para producción**: Deberías procesar el PDF una vez (cuando se sube) y almacenar el retriever en caché, en lugar de reprocesarlo en cada pregunta. Lo mantenemos simple aquí con fines educativos.

---
<a id='section11'></a>
## 11. Comparación Completa del Código Antes y Después

Pongámoslo todo junto. Aquí está la comparación completa del archivo `routers/pdfs.py`.

### ANTES: `routers/pdfs.py` Completo (RAG Regular)

```python
from typing import List
from sqlalchemy.orm import Session
from fastapi import APIRouter, Depends, HTTPException, status, UploadFile, File
import schemas
import crud
from database import SessionLocal
from uuid import uuid4

# Imports de LangChain
import tempfile
import os
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

router = APIRouter(prefix="/pdfs")

# ... (Los endpoints CRUD se mantienen exactamente iguales) ...

def download_pdf_from_url(url: str) -> str:
    response = requests.get(url)
    response.raise_for_status()
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    temp_file.write(response.content)
    temp_file.close()
    return temp_file.name


def create_pdf_rag_agent(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, chunk_overlap=200, add_start_index=True
    )
    all_splits = text_splitter.split_documents(documents)

    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(documents=all_splits)

    @tool
    def search_pdf(query: str) -> str:
        """Search the PDF document for relevant information."""
        results = vector_store.similarity_search(query, k=3)
        return "\n\n".join([doc.page_content for doc in results])

    agent = create_agent(
        model="gpt-4o-mini",
        tools=[search_pdf],
        system_prompt="""You are a helpful assistant that answers questions
        about a PDF document. Use the search_pdf tool to find relevant
        information. Always base your answers on the document."""
    )
    return agent


@router.post("/{id}/ask")
async def ask_pdf(id: int, request: schemas.QuestionRequest, 
                  db: Session = Depends(get_db)):
    pdf = crud.read_pdf(db, id)
    if pdf is None:
        raise HTTPException(status_code=404, detail="PDF not found")

    temp_path = None
    try:
        temp_path = download_pdf_from_url(pdf.file)
        agent = create_pdf_rag_agent(temp_path)
        response = agent.invoke(
            {"messages": [HumanMessage(content=request.question)]}
        )
        answer = response["messages"][-1].content
        return {
            "pdf_id": id, "pdf_name": pdf.name,
            "question": request.question, "answer": answer
        }
    finally:
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)
```

### DESPUÉS: `routers/pdfs.py` Completo (RAG Multimodal)

```python
from typing import List
from sqlalchemy.orm import Session
from fastapi import APIRouter, Depends, HTTPException, status, UploadFile, File
import schemas
import crud
from database import SessionLocal
from uuid import uuid4

# Imports de LangChain (ACTUALIZADOS para multimodal)
import tempfile
import os
import uuid
import base64
import requests
import fitz  # pymupdf: renderiza páginas PDF como imágenes
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.schema.document import Document
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from unstructured_client import UnstructuredClient
from config import get_settings

router = APIRouter(prefix="/pdfs")

# ... (Los endpoints CRUD se mantienen exactamente iguales) ...


def download_pdf_from_url(url: str) -> str:
    """Descargar un PDF desde la URL de S3. (Igual que antes - no se necesitan cambios.)"""
    response = requests.get(url)
    response.raise_for_status()
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    temp_file.write(response.content)
    temp_file.close()
    return temp_file.name


# ========== NUEVO: EXTRACCIÓN DE PDF mediante API de Unstructured + pymupdf ==========

def extract_pdf_elements(pdf_path: str):
    """
    Extraer texto, tablas e imágenes de un archivo PDF.

    - Texto y tablas: extraídos mediante la API de Unstructured (buena para contenido estructurado)
    - Imágenes de páginas: renderizadas mediante pymupdf y enviadas a GPT-4o-mini para su descripción

    pymupdf renderiza cada página como una imagen, lo que garantiza que GPT-4o-mini vea todo
    en la página — incluyendo gráficos basados en vectores que la API de Unstructured puede pasar por alto.
    """
    settings = get_settings()
    client = UnstructuredClient(api_key_auth=settings.UNSTRUCTURED_API_KEY)

    with open(pdf_path, "rb") as f:
        file_content = f.read()

    # Usar la API de Unstructured para extracción de texto y tablas
    response = client.general.partition(
        request={
            "partition_parameters": {
                "files": {
                    "content": file_content,
                    "file_name": os.path.basename(pdf_path),
                },
                "strategy": "hi_res",
                "infer_table_structure": True,
            }
        }
    )

    text_types = {
        "NarrativeText", "Title", "UncategorizedText",
        "ListItem", "Header", "Footer", "FigureCaption",
    }

    raw_texts = []
    table_elements = []

    for element in response.elements:
        el_type = element.get("type", "")
        text = element.get("text", "")

        if el_type in text_types and text.strip():
            raw_texts.append(text)
        elif el_type == "Table":
            table_elements.append(text)

    # Agrupar elementos de texto pequeños en fragmentos de ~2000 caracteres
    text_elements = []
    current_chunk = ""
    for text in raw_texts:
        if len(current_chunk) + len(text) > 2000 and current_chunk:
            text_elements.append(current_chunk.strip())
            current_chunk = text
        else:
            current_chunk += "\n\n" + text if current_chunk else text
    if current_chunk.strip():
        text_elements.append(current_chunk.strip())

    # Renderizar cada página del PDF como imagen usando pymupdf
    # Esto captura todo lo visible en la página: gráficos, diagramas, fotos, etc.
    image_base64_list = []
    doc = fitz.open(pdf_path)
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        img_b64 = base64.b64encode(img_bytes).decode("utf-8")
        image_base64_list.append(img_b64)
    doc.close()

    return text_elements, table_elements, image_base64_list


# ========== NUEVO: FUNCIONES DE RESUMEN ==========

def summarize_image(image_base64: str, model: ChatOpenAI) -> str:
    """Usar un modelo con capacidad de visión para describir una imagen de página (PNG en base64 de pymupdf)."""
    message = HumanMessage(content=[
        {"type": "text", "text": """Describe this image in detail. Include:
                - What type of content it shows (chart, diagram, photo, etc.)
                - Any text visible in the image
                - Key data points if it's a chart or graph
                - The overall meaning or purpose of the image"""},
        {"type": "image_url", "image_url": {
            "url": f"data:image/jpeg;base64,{image_base64}"
        }}
    ])
    return model.invoke([message]).content


def summarize_text(text: str, model: ChatOpenAI) -> str:
    """Resumir un elemento de texto."""
    prompt = f"Summarize concisely:\n\n{text}\n\nSummary:"
    return model.invoke([HumanMessage(content=prompt)]).content


def summarize_table(table: str, model: ChatOpenAI) -> str:
    """Resumir un elemento de tabla."""
    prompt = f"Summarize this table with key data points:\n\n{table}\n\nSummary:"
    return model.invoke([HumanMessage(content=prompt)]).content


def create_all_summaries(text_elements, table_elements, image_base64_list):
    """Crear resúmenes para todos los tipos de contenido usando gpt-4o-mini."""
    model = ChatOpenAI(model="gpt-4o-mini", max_tokens=1024)

    text_summaries = [summarize_text(t, model) for t in text_elements]
    table_summaries = [summarize_table(t, model) for t in table_elements]
    image_summaries = [summarize_image(b64, model) for b64 in image_base64_list]

    return text_summaries, table_summaries, image_summaries


# ========== NUEVO: MULTI-VECTOR RETRIEVER ==========

def create_multimodal_retriever(
    text_summaries, text_elements,
    table_summaries, table_elements,
    image_summaries
):
    """Crear un Multi-Vector Retriever para contenido multimodal."""
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = Chroma(
        collection_name="multimodal_summaries",
        embedding_function=embeddings
    )
    docstore = InMemoryStore()
    id_key = "doc_id"

    retriever = MultiVectorRetriever(
        vectorstore=vectorstore, docstore=docstore, id_key=id_key
    )

    def add_docs(summaries, originals, content_type):
        if not summaries:
            return
        doc_ids = [str(uuid.uuid4()) for _ in summaries]
        summary_docs = [
            Document(page_content=s, metadata={id_key: doc_ids[i], "type": content_type})
            for i, s in enumerate(summaries)
        ]
        retriever.vectorstore.add_documents(summary_docs)
        retriever.docstore.mset(list(zip(doc_ids, originals)))

    add_docs(text_summaries, text_elements, "text")
    add_docs(table_summaries, table_elements, "table")
    add_docs(image_summaries, image_summaries, "image")

    return retriever


# ========== NUEVO: AGENTE MULTIMODAL ==========

def create_multimodal_rag_agent(retriever):
    """Crear un agente RAG que sabe sobre texto, tablas e imágenes."""

    @tool
    def search_documents(query: str) -> str:
        """Search documents for text, table data, and image descriptions."""
        results = retriever.invoke(query)
        if not results:
            return "No relevant information found."
        formatted = []
        for i, r in enumerate(results, 1):
            content = r.page_content if hasattr(r, 'page_content') else str(r)
            formatted.append(f"[Result {i}]\n{content}")
        return "\n\n".join(formatted)

    agent = create_agent(
        model="gpt-4o-mini",
        tools=[search_documents],
        system_prompt="""You are a helpful assistant that answers questions 
        about documents. You have access to text, tables, and image 
        descriptions. Use search_documents to find relevant information. 
        Base your answers ONLY on retrieved information."""
    )
    return agent


# ========== ENDPOINT /ask ACTUALIZADO ==========

@router.post("/{id}/ask")
async def ask_pdf(id: int, request: schemas.QuestionRequest,
                  db: Session = Depends(get_db)):
    pdf = crud.read_pdf(db, id)
    if pdf is None:
        raise HTTPException(status_code=404, detail="PDF not found")

    temp_path = None
    try:
        temp_path = download_pdf_from_url(pdf.file)

        text_el, table_el, image_b64 = extract_pdf_elements(temp_path)
        text_sum, table_sum, image_sum = create_all_summaries(
            text_el, table_el, image_b64
        )
        retriever = create_multimodal_retriever(
            text_sum, text_el, table_sum, table_el, image_sum
        )
        agent = create_multimodal_rag_agent(retriever)

        response = agent.invoke(
            {"messages": [HumanMessage(content=request.question)]}
        )
        answer = response["messages"][-1].content

        return {
            "pdf_id": id, "pdf_name": pdf.name,
            "question": request.question, "answer": answer
        }
    finally:
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)
```

---
### ¿Qué Pasa con los Imports?

Aquí tienes una comparación clara de qué imports cambiaron:

```python
# ========== IMPORTS ELIMINADOS ==========
# Estos ya no son necesarios:
from langchain_community.document_loaders import PyPDFLoader      # Reemplazado por la API de Unstructured
from langchain_text_splitters import RecursiveCharacterTextSplitter # Reemplazado por agrupación local
from langchain_core.vectorstores import InMemoryVectorStore        # Reemplazado por MultiVectorRetriever

# ========== IMPORTS AÑADIDOS ==========
# Estos son nuevos:
import uuid                                                                    # Para IDs únicos
import base64                                                                  # Para codificar imágenes de páginas
import fitz                                                                    # pymupdf: renderiza páginas PDF como imágenes
import requests                                                                # Para descargar PDFs desde S3
from langchain_openai import ChatOpenAI                                        # Para modelos de resumen
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever     # Nuevo retriever
from langchain_classic.schema.document import Document                         # Para crear documentos
from langchain_classic.storage import InMemoryStore                            # Doc store para originales
from langchain_chroma import Chroma                                            # Base de datos vectorial
from unstructured_client import UnstructuredClient                             # Cliente API para extracción de texto/tablas
from config import get_settings                                                # Para acceder a UNSTRUCTURED_API_KEY
```

### ¿Por Qué `langchain_classic`?

En LangChain 1.x, varios módulos se movieron del paquete `langchain` a `langchain_classic`. Tres de nuestros imports se ven afectados:

| Clase | Ruta antigua (no funciona en 1.x) | Ruta correcta |
|-------|-----------------------------------|----------------|
| `MultiVectorRetriever` | `langchain.retrievers.multi_vector` | `langchain_classic.retrievers.multi_vector` |
| `Document` | `langchain.schema.document` | `langchain_classic.schema.document` |
| `InMemoryStore` | `langchain.storage` | `langchain_classic.storage` |

Los otros imports (`tool`, `create_agent`, `HumanMessage`) siguen funcionando bien desde `langchain`.

### Lo Que Ya NO Se Necesita

Varios imports de los enfoques anteriores ya no son necesarios:

| Ya no se necesita | Por qué |
|-------------------|----------|
| `import shutil` | Se usaba para limpiar directorios temporales de imágenes. Las imágenes de páginas se generan en memoria con pymupdf. |
| `from pathlib import Path` | Se usaba para crear el directorio de salida de imágenes. No se necesitan directorios temporales. |
| `from unstructured.partition.pdf import partition_pdf` | Reemplazado por llamadas a la API de `UnstructuredClient` para texto/tablas, y pymupdf para imágenes. |
| `PyPDFLoader` | Reemplazado por `UnstructuredClient` para texto/tablas. |
| `RecursiveCharacterTextSplitter` | Reemplazado por agrupación local en fragmentos de ~2000 caracteres. |

---
<a id='section12'></a>
## 12. Resumen de Todos los Cambios

### Lista de Verificación Completa

Aquí tienes cada cambio necesario para transformar la app:

| # | Archivo | Cambio | Dificultad |
|---|---------|--------|------------|
| 1 | `pyproject.toml` | Añadir `unstructured-client`, `langchain-unstructured`, `langchain-chroma`, `pillow`, `pymupdf` | Fácil |
| 2 | `.env` | Añadir `UNSTRUCTURED_API_KEY` | Fácil |
| 3 | `config.py` | Añadir `UNSTRUCTURED_API_KEY: str` a la clase Settings | Fácil |
| 4 | `routers/pdfs.py` | Actualizar imports (añadir `base64`, `fitz`, `UnstructuredClient`, etc.) | Fácil |
| 5 | `routers/pdfs.py` | Añadir función `extract_pdf_elements()` (API de Unstructured para texto/tablas + pymupdf para imágenes de páginas) | Medio |
| 6 | `routers/pdfs.py` | Añadir funciones de resumen (`summarize_image`, `summarize_text`, `summarize_table`, `create_all_summaries`) | Medio |
| 7 | `routers/pdfs.py` | Añadir función `create_multimodal_retriever()` | Medio |
| 8 | `routers/pdfs.py` | Reemplazar `create_pdf_rag_agent()` por `create_multimodal_rag_agent()` | Fácil |
| 9 | `routers/pdfs.py` | Actualizar el endpoint `/ask` | Fácil |
| 10 | Frontend (opcional) | Actualizar texto del placeholder | Trivial |

### Archivos Que NO Cambian

| Archivo | Por Qué No Hay Cambios |
|---------|------------------------|
| `main.py` | Solo importa e inicia el servidor |
| `database.py` | La conexión a la base de datos sigue igual |
| `models.py` | La estructura de la tabla PDF sigue igual |
| `schemas.py` | El formato de solicitud/respuesta sigue igual |
| `crud.py` | Las operaciones CRUD siguen iguales |
| Frontend `pdf.js` | Misma interfaz de preguntas y respuestas |
| Frontend `pdf-list.js` | Misma lista de PDFs |
| Frontend estilos | Mismos estilos |

### Archivos Que SÍ Cambian

| Archivo | Qué Cambió |
|---------|-------------|
| `pyproject.toml` | Se añadió `unstructured-client`, `langchain-unstructured`, `langchain-chroma`, `pillow`, `pymupdf` |
| `.env` | Se añadió `UNSTRUCTURED_API_KEY` |
| `config.py` | Se añadió la configuración `UNSTRUCTURED_API_KEY: str` |
| `routers/pdfs.py` | Reescritura importante de la lógica RAG multimodal (ver secciones anteriores) |

### El Flujo de Datos: Antes vs Después

**ANTES (RAG Regular)**:
```
PDF --> PyPDFLoader --> Fragmentos de texto --> Embeddings --> InMemoryVectorStore --> Agente --> Respuesta
```

**DESPUÉS (RAG Multimodal)**:
```
         +-> API de Unstructured -+-- Texto  --> Resumir (GPT-4o-mini) --+
         |                        |                                      |
PDF -----+                        +-- Tablas --> Resumir (GPT-4o-mini) --+--> MultiVectorRetriever --> Agente --> Respuesta
         |                                                               |
         +-> pymupdf (páginas) --- Imágenes --> Describir (GPT-4o-mini) -+
```

### Lo Que Experimenta el Usuario

Desde la perspectiva del usuario, ¡la app se ve exactamente igual! La única diferencia está en lo que puede responder:

| Pregunta | Antes (RAG Regular) | Después (RAG Multimodal) |
|----------|---------------------|---------------------------|
| "¿De qué trata este documento?" | Puede responder | Puede responder |
| "¿Cuáles son los gastos totales?" | Puede responder (si está en el texto) | Puede responder (incluso si está en una tabla) |
| "¿Qué muestra el gráfico?" | **No puede responder** | ¡Puede responder! |
| "Describe el diagrama de la página 5" | **No puede responder** | ¡Puede responder! |
| "¿Qué producto se muestra en las imágenes?" | **No puede responder** | ¡Puede responder! |

---
## Conclusiones Clave

1. **El frontend no necesita cambiar** porque simplemente envía preguntas y recibe respuestas en texto. La magia multimodal ocurre completamente en el backend.

2. **El cambio principal en el backend usa un enfoque de dos herramientas para la extracción de PDFs**: la API de Unstructured (mediante `unstructured-client`) extrae texto y tablas, mientras que `pymupdf` renderiza cada página como una imagen. Esta combinación garantiza que nada se pierda — incluso gráficos basados en vectores que la API de Unstructured no puede detectar como imágenes.

3. **Las imágenes de páginas se manejan describiéndolas en texto** usando las capacidades de visión de GPT-4o. pymupdf renderiza cada página como un PNG, lo codifica en base64 y lo envía a GPT-4o para obtener una descripción detallada.

4. **El Multi-Vector Retriever usa dos almacenes**: resúmenes para buscar (rápido) y originales para responder (preciso).

5. **La mayoría de los archivos no cambian en absoluto**: `main.py`, `database.py`, `models.py`, `schemas.py`, `crud.py` se mantienen exactamente iguales. Los cambios están en `config.py` (nueva configuración de clave API), `routers/pdfs.py` (lógica RAG multimodal) y `pyproject.toml` (nuevas dependencias incluyendo `pymupdf`).

6. **El contrato de la API se mantiene igual**: El endpoint `/pdfs/{id}/ask` sigue aceptando `{"question": "..."}` y devolviendo `{"answer": "..."}`. Por eso el frontend no necesita cambios.

7. **En producción**, querrías procesar los PDFs en el momento de la subida y almacenar en caché el retriever, en lugar de reprocesar en cada pregunta. Esta es una optimización que se deja como ejercicio.

---
## ¡Enhorabuena!

¡Ahora entiendes cómo actualizar una aplicación RAG full-stack para manejar contenido multimodal! La idea clave es que **una buena arquitectura de software facilita las actualizaciones**: como nuestra app tenía una separación limpia entre frontend y backend, y un contrato de API bien definido, solo necesitamos cambiar la implementación interna de un archivo (`routers/pdfs.py`) para añadir una nueva capacidad potente.

Este es un principio fundamental de la ingeniería de software: **programa hacia interfaces, no hacia implementaciones**. El frontend programa hacia la interfaz de la API `/ask`, así que no le importa si la implementación detrás de ella cambia de RAG regular a RAG multimodal.